# PyTorch resnet18 Relax

In [1]:
import torch
from torchvision import models
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_fx

fold_pipeline = tvm.transform.Sequential([
    relax.transform.FoldBatchnormToConv2D(),
    relax.transform.FoldConstant(),
])

# 创建 PyTorch 模型
torch_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).eval()
shape = [1, 3, 224, 224]
input_info = [(shape, "float32")]
# 变换为 Relay 模型
with torch.no_grad():
    graph_model = torch.fx.symbolic_trace(torch_model)
    mod = from_fx(graph_model, input_info)
# 初次优化模型
run_mod = fold_pipeline(mod)

In [2]:
run_mod.show()

In [5]:
from tvm.relax.dpl.pattern import wildcard, is_op, is_const, make_fused_bias_activation_pattern

In [ ]:
compiler = "ccompiler"
conv2d_bias_relu_pat = make_fused_bias_activation_pattern(
    "relax.nn.conv2d",
    with_bias=True,
    activation="relax.nn.relu"
)
conv2d_relu_pat = make_fused_bias_activation_pattern(
    "relax.nn.conv2d",
    with_bias=False,
    activation="relax.nn.relu"
)
conv2d_bias_pat = make_fused_bias_activation_pattern(
    "relax.nn.conv2d",
    with_bias=True,
)
patterns = [
    (f"{compiler}.conv2d_bias_relu", conv2d_bias_relu_pat),
    (f"{compiler}.conv2d_relu", conv2d_relu_pat),
    (f"{compiler}.conv2d_bias", conv2d_bias_pat),
]
fuse_pipeline = tvm.transform.Sequential([
    relax.transform.FuseOpsByPattern(patterns, bind_constants=True),
    # relax.transform.FuseTransposeMatmul(),
    # relax.transform.FuseOps(),
    # relax.transform.FuseTIR(),
])
run_mod2 = fuse_pipeline(run_mod)

In [16]:
run_mod2.show()

In [17]:
test_mod = tvm.IRModule({"main": run_mod2["fused_relax_nn_conv2d_relax_add"]})
test_mod = fold_pipeline(test_mod)

In [18]:
test_mod.show()